In [1]:
!pip3 install kafka-python

     |████████████████████████████████| 276 kB 2.3 MB/s eta 0:00:01


In [2]:
!nc -vz spark-master 7077

Connection to spark-master 7077 port [tcp/*] succeeded!


In [3]:
!ls

PipelineTest.ipynb  Untitled.ipynb


In [4]:
!find spark-jars -type f -name "*.jar"


find: ‘spark-jars’: No such file or directory


In [5]:
import requests
import json
from kafka import KafkaProducer

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [6]:
def produce_api_data_to_kafka(kafka_bootstrap_servers):
    # Lấy dữ liệu từ API
    api_url = ("https://www.pegelonline.wsv.de/webservices/rest-api/v2/stations.json?"
               "includeTimeseries=true&hasTimeseries=WV&includeForecastTimeseries=true")
    response = requests.get(api_url)
    if response.status_code != 200:
        print("Failed to fetch API data, status code:", response.status_code)
        return
    stations = response.json()

    producer = KafkaProducer(
        bootstrap_servers=kafka_bootstrap_servers,
        value_serializer=lambda v: json.dumps(v).encode('utf-8')
    )

    for station in stations:
        station_data = {
            "uuid": station.get("uuid"),
            "number": station.get("number"),
            "shortname": station.get("shortname"),
            "longname": station.get("longname"),
            "km": station.get("km"),
            "agency": station.get("agency"),
            "longitude": station.get("longitude"),
            "latitude": station.get("latitude"),
            "water": station.get("water")
        }
        timeseries_list = station.get("timeseries", [])
        for ts in timeseries_list:
            record = {
                "station": station_data,
                "timeseries": ts
            }
            topic = "timeseries_" + ts.get("shortname", "unknown")
            producer.send(topic, record)
    producer.flush()
    print("API data produced to Kafka topics.")

print("Function produce_api_data_to_kafka defined")

Function produce_api_data_to_kafka defined


In [7]:
kafka_bootstrap_servers = ["kafka:9092"]

produce_api_data_to_kafka(kafka_bootstrap_servers)

API data produced to Kafka topics.


In [8]:
from pyspark import SparkContext
if SparkContext._active_spark_context:
    SparkContext.getOrCreate().stop()

In [9]:
# spark = SparkSession.builder \
#     .appName("PipelineTest") \
#     .master("spark://spark-master:7077") \
#     .config("spark.sql.warehouse.dir", "hdfs://namenode:8020/user/hive/warehouse") \
#     .config("spark.metrics.conf", "") \
#     .getOrCreate()
# spark.conf.set("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
# spark.conf.set("spark.hadoop.fs.s3a.access.key", "test")
# spark.conf.set("spark.hadoop.fs.s3a.secret.key", "12345678")
# spark.conf.set("spark.hadoop.fs.s3a.path.style.access", "true")
# spark.conf.set("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
# spark.conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
# spark.conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

In [10]:
spark = SparkSession.builder \
    .appName("PipelineTest") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:8020/user/hive/warehouse") \
    .getOrCreate()

In [11]:
sc = spark.sparkContext
    
sc._jsc.hadoopConfiguration().set("fs.s3a.access.key", "test")
sc._jsc.hadoopConfiguration().set("fs.s3a.secret.key", "12345678")
sc._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "http://localhost:9000")
sc._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
sc._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "false")
sc._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
sc._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "false")

In [12]:
for k, v in spark.sparkContext.getConf().getAll():
    if k.startswith("spark.jars"):
        print(f"{k} = {v}")

In [13]:
classpath = spark.sparkContext._gateway.jvm.java.lang.System.getProperty("java.class.path")
print("Classpath:", classpath)

Classpath: /usr/local/spark/conf/:/usr/local/spark/jars/arrow-vector-2.0.0.jar:/usr/local/spark/jars/hadoop-mapreduce-client-jobclient-3.2.0.jar:/usr/local/spark/jars/zjsonpatch-0.3.0.jar:/usr/local/spark/jars/machinist_2.12-0.6.8.jar:/usr/local/spark/jars/compress-lzf-1.0.3.jar:/usr/local/spark/jars/breeze-macros_2.12-1.0.jar:/usr/local/spark/jars/logging-interceptor-3.12.12.jar:/usr/local/spark/jars/nimbus-jose-jwt-4.41.1.jar:/usr/local/spark/jars/commons-codec-1.10.jar:/usr/local/spark/jars/jackson-jaxrs-base-2.9.5.jar:/usr/local/spark/jars/commons-math3-3.4.1.jar:/usr/local/spark/jars/aircompressor-0.10.jar:/usr/local/spark/jars/okhttp-3.12.12.jar:/usr/local/spark/jars/hadoop-auth-3.2.0.jar:/usr/local/spark/jars/jcl-over-slf4j-1.7.30.jar:/usr/local/spark/jars/jackson-annotations-2.10.0.jar:/usr/local/spark/jars/spire-macros_2.12-0.17.0-M1.jar:/usr/local/spark/jars/spark-launcher_2.12-3.1.1.jar:/usr/local/spark/jars/aopalliance-1.0.jar:/usr/local/spark/jars/kerby-config-1.0.1.jar:/u

In [14]:
station_schema = StructType([
    StructField("uuid", StringType()),
    StructField("number", StringType()),
    StructField("shortname", StringType()),
    StructField("longname", StringType()),
    StructField("km", DoubleType()),
    StructField("agency", StringType()),
    StructField("longitude", DoubleType()),
    StructField("latitude", DoubleType()),
    StructField("water", StructType([
        StructField("shortname", StringType()),
        StructField("longname", StringType())
    ]))
])

timeseries_schema = StructType([
    StructField("shortname", StringType()),
    StructField("longname", StringType()),
    StructField("unit", StringType()),
    StructField("equidistance", IntegerType()),
    StructField("gaugeZero", StructType([
        StructField("unit", StringType()),
        StructField("value", DoubleType()),
        StructField("validFrom", StringType())
    ]), True),
    StructField("start", StringType(), True),
    StructField("end", StringType(), True)
])

full_schema = StructType([
    StructField("station", station_schema),
    StructField("timeseries", timeseries_schema)
])

print("Schema defined")

Schema defined


In [15]:
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", ",".join(kafka_bootstrap_servers)) \
    .option("subscribePattern", "timeseries_.*") \
    .option("startingOffsets", "earliest") \
    .load()

kafka_df = kafka_df.selectExpr("CAST(value AS STRING) as value", "topic", "timestamp")
processed_df = kafka_df.withColumn("json_data", from_json(col("value"), full_schema)) \
    .select("topic", "timestamp", "json_data.*")

display(processed_df)

DataFrame[topic: string, timestamp: timestamp, station: struct<uuid:string,number:string,shortname:string,longname:string,km:double,agency:string,longitude:double,latitude:double,water:struct<shortname:string,longname:string>>, timeseries: struct<shortname:string,longname:string,unit:string,equidistance:int,gaugeZero:struct<unit:string,value:double,validFrom:string>,start:string,end:string>]

In [16]:
consoleQuery = processed_df.writeStream \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime="10 seconds") \
    .start()

print("Streaming query started. Xem log trong cell Notebook hoặc console của Spark UI (port 4040).")

Streaming query started. Xem log trong cell Notebook hoặc console của Spark UI (port 4040).


In [17]:
hdfsParquetQuery = processed_df.writeStream \
   .format("parquet") \
   .option("path", "hdfs://namenode:8020/youruser/sensor_data_parquet") \
   .option("checkpointLocation", "/tmp/checkpoint/hdfs_sensor_data") \
   .trigger(processingTime="10 seconds") \
   .start()

In [ ]:
minioParquetQuery = processed_df.writeStream \
   .format("parquet") \
   .option("path", "s3a://minio-test/sensor_data_parquet") \
   .option("checkpointLocation", "/tmp/checkpoint/minio_sensor_data") \
   .trigger(processingTime="10 seconds") \
   .start()

In [ ]:
!pip install minio  # nếu chưa cài

from minio import Minio
from minio.error import S3Error

# Khởi tạo client, đảm bảo rằng endpoint phù hợp với cấu hình của container minio
client = Minio(
    "minio:9000",  # hostname của container minio trong mạng docker
    access_key="test",
    secret_key="12345678",
    secure=False  # vì chúng ta không dùng SSL
)

# Kiểm tra kết nối bằng cách gọi hàm bucket_exists
bucket_name = "minio-test"  # tên bucket bạn muốn kiểm tra
try:
    exists = client.bucket_exists(bucket_name)
    if exists:
        print(f"Bucket '{bucket_name}' đã tồn tại.")
    else:
        print(f"Bucket '{bucket_name}' chưa tồn tại. Tạo bucket...")
        client.make_bucket(bucket_name)
        print(f"Bucket '{bucket_name}' được tạo thành công.")
except S3Error as err:
    print("Lỗi kết nối hoặc thao tác với MinIO:", err)


In [ ]:
# spark.stop()